# Role Explorer

Select a Role value to see the activity sequences associated with it.

In [ ]:
# Setup
import sys
from pathlib import Path
from collections import defaultdict
import ipywidgets as widgets
from IPython.display import display, HTML
import torch

_current = Path().resolve()
while _current != _current.parent:
    if (_current / 'src').is_dir():
        break
    _current = _current.parent

if str(_current) not in sys.path:
    sys.path.insert(0, str(_current))
if str(_current / 'src') not in sys.path:
    sys.path.insert(0, str(_current / 'src'))

from src.interpretability.config.domestic_declarations_config import CONFIG
from src.model.dropout_uncertainty_enc_dec_LSTM.dropout_uncertainty_model import DropoutUncertaintyEncoderDecoderLSTM

In [ ]:
# Load model and test dataset
model = DropoutUncertaintyEncoderDecoderLSTM.load(str(CONFIG.get_model_path()), dropout=0.0)
test_dataset = torch.load(str(CONFIG.get_test_data_path()), weights_only=False)

# Get category mappings from model (not from CONFIG which may be limited)
cat_features = model.data_set_categories[0]
activity_mapping = None
role_mapping = None
activity_feat_idx = None
role_feat_idx = None

for idx, (feat_name, num_values, mapping) in enumerate(cat_features):
    if feat_name == CONFIG.concept_name:
        activity_mapping = {v: k for k, v in mapping.items()}
        activity_feat_idx = idx
    if feat_name == 'Role':
        role_mapping = {v: k for k, v in mapping.items()}
        role_feat_idx = idx

print(f"Loaded {len(test_dataset)} samples")
print(f"Activity feature index: {activity_feat_idx}")
print(f"Role feature index: {role_feat_idx}")
if role_feat_idx is None:
    print("WARNING: 'Role' not found in model categories!")

In [ ]:
# Extract full cases (group samples by case name, take the longest/full case)
# Each sample in test_dataset is (cat_tensors, num_tensors, case_name)

case_data = {}  # case_name -> (role, activity_sequence)

for sample in test_dataset:
    cat_tensors, num_tensors, case_name = sample
    
    # Get activity sequence (skip padding - index 0)
    activity_tensor = cat_tensors[activity_feat_idx].squeeze()
    activities = []
    for idx in activity_tensor.tolist():
        if idx == 0:  # Skip padding
            continue
        act_name = activity_mapping.get(idx, f'Unknown_{idx}')
        if act_name not in ['EOS', 'PAD', '<EOS>', '<PAD>']:
            activities.append(act_name)
    
    # Get role (from first non-padding position)
    if role_feat_idx is not None and role_mapping is not None:
        role_tensor = cat_tensors[role_feat_idx].squeeze()
        # Find first non-zero (non-padding) value
        role_name = 'Unknown'
        for r_idx in role_tensor.tolist():
            if r_idx != 0:
                role_name = role_mapping.get(r_idx, f'Unknown_{r_idx}')
                break
    else:
        role_name = 'N/A'
    
    activity_seq = tuple(activities)
    
    # Keep only the longest sequence per case (full case)
    if case_name not in case_data or len(activity_seq) > len(case_data[case_name][1]):
        case_data[case_name] = (role_name, activity_seq)

print(f"Found {len(case_data)} full cases")

# Show sample of roles found
sample_roles = set(r for r, _ in case_data.values())
print(f"Sample roles: {sorted(list(sample_roles))[:10]}")

In [ ]:
# Build mapping: role -> {activity_sequence: count}
role_to_sequences = defaultdict(lambda: defaultdict(int))

for case_name, (role, activity_seq) in case_data.items():
    role_to_sequences[role][activity_seq] += 1

# Get ALL roles from the model's vocabulary (not just those in test data)
all_roles = []
if role_mapping is not None:
    all_roles = list(role_mapping.values())

# Sort roles alphabetically
sorted_roles = sorted(all_roles)
roles_in_data = set(role_to_sequences.keys())

print(f"Total roles in model vocabulary: {len(sorted_roles)}")
print(f"Roles found in test data: {len(roles_in_data)}")
print(f"Roles without test cases: {len(sorted_roles) - len(roles_in_data)}")

## Select Role to Explore

In [ ]:
# Create dropdown and output
role_dropdown = widgets.Dropdown(
    options=sorted_roles,
    value=sorted_roles[0] if sorted_roles else None,
    description='Role:',
    style={'description_width': 'initial'}
)

output = widgets.Output()

def on_role_change(change):
    with output:
        output.clear_output(wait=True)
        role = change['new']
        sequences = role_to_sequences[role]
        
        print(f"Role: {role}")
        
        if not sequences:
            print("=" * 80)
            print("\n  No cases with this role in the test dataset.")
            print("    This role exists in the model vocabulary but has no test examples.")
            return
        
        # Sort sequences by count (descending)
        sorted_seqs = sorted(sequences.items(), key=lambda x: x[1], reverse=True)
        total_cases = sum(sequences.values())
        
        print(f"Total cases: {total_cases}")
        print("=" * 80)
        
        if len(sorted_seqs) > 1:
            print(f"\n  NOTE: {len(sorted_seqs)} different activity sequences share this role.\n")
        
        for i, (seq, count) in enumerate(sorted_seqs):
            print(f"\nSequence {i+1} ({count} cases):")
            print("-" * 40)
            for j, act in enumerate(seq):
                print(f"  {j+1}. {act}")

role_dropdown.observe(on_role_change, names='value')

# Initial display
display(role_dropdown)
display(output)

# Trigger initial update
on_role_change({'new': role_dropdown.value})